In [1]:
from typing import TypedDict, Annotated
from dotenv import load_dotenv

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")


class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


def chat_node(state: ChatState):
    decision = interrupt({
        "type": "approval",
        "reason": "Confirm before sending to the LLM",
        "question": state["messages"][-1].content,
        "instruction": "Approve or reject (approved: true/false)",
    })

    if decision.get("approved") is False:
        return {"messages": [AIMessage(content="Not approved.")]}

    response = llm.invoke(state["messages"])
    return {"messages": [response]}


graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

checkpointer = InMemorySaver()
chatbot = graph.compile(checkpointer=checkpointer)

In [2]:
config = {"configurable": {"thread_id": "1"}}

# FIRST invoke — runs until interrupt, then pauses
initial = {"messages": [HumanMessage(content="Explain gradient descent simply")]}
result = chatbot.invoke(initial, config=config)

print(result["__interrupt__"])   # the payload you sent to the "frontend"

[Interrupt(value={'type': 'approval', 'reason': 'Confirm before sending to the LLM', 'question': 'Explain gradient descent simply', 'instruction': 'Approve or reject (approved: true/false)'}, id='bdf6ddf57c10a0326467c89128450e62')]


In [3]:
# Show the interrupt to the human, get their answer
user_input = input("Approve? (yes/no): ")
approved = user_input.strip().lower() == "yes"

# SECOND invoke — resume with the human's decision
final = chatbot.invoke(
    Command(resume={"approved": approved}),
    config=config
)
print(final["messages"][-1].content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text', 'text': 'Imagine you are trapped on top of a foggy mountain, and you want to get down to the valley at the very bottom. \n\nBecause the fog is so thick, **you can’t see the bottom of the mountain.** You can only see the ground right beneath your feet.\n\nHow do you get down?\n\n1. You feel the slope of the ground with your foot.\n2. You find the direction that slopes **downward the most**.\n3. You take a step in that direction.\n4. You repeat this process, step by step, until the ground feels completely flat. \n\nCongratulations! You’ve reached the bottom. \n\nThat process is **Gradient Descent**.\n\n---\n\n### How this works in Machine Learning\n\nIn AI and Machine Learning, the goal of a computer program is to make accurate predictions. When it starts out, it makes a lot of mistakes. \n\nHere is how the mountain analogy maps to a computer learning:\n\n* **The Mountain = Error.** The higher up you are, the more mistakes the computer is making. The bottom of the valle